# 👗 Projeto Ponta a Ponta — TensorFlow + Fashion MNIST

Construção de uma CNN (Rede Neural Convolucional) com **TensorFlow/Keras** para classificar
peças de roupa e acessórios usando o dataset **Fashion MNIST**.

---

## 🗺️ Pipeline do projeto

- [x] 🔧 Configuração do ambiente
- [x] 📦 Imports e configuração visual
- [x] 📂 Carregamento do dataset
- [x] 🔍 Inspeção e visualização dos dados
- [x] ⚙️ Pré-processamento
- [x] 🏗️ Construção do modelo (CNN)
- [x] 🚀 Treinamento
- [x] 📈 Curvas de treinamento
- [x] 🎯 Avaliação (métricas, matriz de confusão, relatório)
- [x] 🔬 Inferência com imagens do dataset
- [x] 🖼️ Teste com imagem local externa
- [x] 🏁 Conclusão e próximos passos

---

> **Dataset:** [Fashion MNIST](https://keras.io/api/datasets/fashion_mnist/)  
> **Referência de arquitetura:** CNN com camadas Conv2D + MaxPooling + Dense


## 🔧 Configuração do Ambiente

Verificação das versões do Python e TensorFlow, silenciamento de logs verbosos
(específico para notebooks) e definição das seeds para reprodutibilidade.


In [ ]:
# ── Versões ──────────────────────────────────────────────────────────────────
from platform import python_version
import os

print('Versão do Python:', python_version())

# TensorFlow normalmente já vem instalado no Google Colab.
# Se necessário: !pip install -q tensorflow
import tensorflow as tf
print('Versão do TensorFlow:', tf.__version__)

# ── Silenciar logs do TensorFlow (apenas em notebooks) ───────────────────────
# TF_CPP_MIN_LOG_LEVEL: 0=DEBUG, 1=INFO, 2=WARNING, 3=ERROR
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

# ── Reprodutibilidade ─────────────────────────────────────────────────────────
import numpy as np

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print(f'Seeds definidas: {SEED}')


## 📦 Imports e Configuração Visual


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from tensorflow.keras import datasets, layers, models, callbacks
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ── Configuração visual padrão ────────────────────────────────────────────────
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110


## 📂 Carregamento do Dataset

O **Fashion MNIST** contém **70 000 imagens** em tons de cinza (28 × 28 px) de peças de roupa,
divididas em **10 classes**:

| # | Classe (EN) | Classe (PT) |
|---|-------------|-------------|
| 0 | T-shirt/Top | Camiseta/Top |
| 1 | Trouser | Calça |
| 2 | Pullover | Pulôver |
| 3 | Dress | Vestido |
| 4 | Coat | Casaco |
| 5 | Sandal | Sandália |
| 6 | Shirt | Camisa |
| 7 | Sneaker | Tênis |
| 8 | Bag | Bolsa |
| 9 | Ankle boot | Bota |

> 💡 **Split padrão:** 60 000 imagens para treino e 10 000 para teste.


In [ ]:
# Carrega o dataset Fashion MNIST — já disponível no Keras
(imagens_treino, labels_treino), (imagens_teste, labels_teste) = datasets.fashion_mnist.load_data()

nomes_classes = [
    'Camiseta/Top', 'Calça', 'Pulôver', 'Vestido', 'Casaco',
    'Sandália', 'Camisa', 'Tênis', 'Bolsa', 'Bota'
]

print('📐 Formato das imagens de treino :', imagens_treino.shape)
print('📐 Formato das labels de treino  :', labels_treino.shape)
print('📐 Formato das imagens de teste  :', imagens_teste.shape)
print('📐 Formato das labels de teste   :', labels_teste.shape)


In [ ]:
def visualiza_imagens(images, labels, class_names, n_linhas=3, n_colunas=5):
    """Exibe uma grade de imagens com seus rótulos.

    Parameters
    ----------
    images : np.ndarray  — imagens no formato (N, H, W) ou (N, H, W, 1)
    labels : np.ndarray  — rótulos inteiros correspondentes
    class_names : list   — lista com os nomes das classes
    n_linhas : int       — número de linhas na grade
    n_colunas : int      — número de colunas na grade
    """
    total = min(n_linhas * n_colunas, len(images))
    fig, axes = plt.subplots(n_linhas, n_colunas, figsize=(n_colunas * 2, n_linhas * 2.2))
    axes = axes.flatten()

    for i in range(total):
        axes[i].imshow(images[i].squeeze(), cmap='gray')
        axes[i].set_title(class_names[int(labels[i])], fontsize=9)
        axes[i].axis('off')

    for j in range(total, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('🔍 Amostras do dataset Fashion MNIST', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


visualiza_imagens(imagens_treino, labels_treino, nomes_classes)


## ⚙️ Pré-processamento

Etapas aplicadas:
1. **Normalização** — pixels de `uint8 [0, 255]` → `float32 [0.0, 1.0]`
2. **Expand dims** — adiciona o canal da imagem: `(N, 28, 28)` → `(N, 28, 28, 1)` (exigido pela Conv2D)

> 💡 Os rótulos (`labels_*`) permanecem inteiros, compatíveis com `sparse_categorical_crossentropy`.


In [ ]:
# Normaliza os valores dos pixels para [0, 1]
imagens_treino = imagens_treino.astype('float32') / 255.0
imagens_teste  = imagens_teste.astype('float32')  / 255.0

# Adiciona o canal: (N, 28, 28) → (N, 28, 28, 1)
imagens_treino = np.expand_dims(imagens_treino, axis=-1)
imagens_teste  = np.expand_dims(imagens_teste,  axis=-1)

print('✅ Novo formato das imagens de treino:', imagens_treino.shape)
print('✅ Novo formato das imagens de teste :', imagens_teste.shape)


## 🏗️ Construção do Modelo

Arquitetura **CNN** com:
- Bloco de **Feature Learning**: três camadas Conv2D + MaxPooling
- Bloco de **Classificação**: Flatten → Dense → Dropout → Softmax (10 classes)


In [ ]:
# ── Arquitetura do modelo (Feature Learning + Classificação) ─────────────────
modelo_lia = models.Sequential(name='CNN_FashionMNIST')

# Bloco 1
modelo_lia.add(layers.Input(shape=(28, 28, 1)))
modelo_lia.add(layers.Conv2D(32, (3, 3), activation='relu'))
modelo_lia.add(layers.MaxPooling2D((2, 2)))

# Bloco 2
modelo_lia.add(layers.Conv2D(64, (3, 3), activation='relu'))
modelo_lia.add(layers.MaxPooling2D((2, 2)))

# Bloco 3
modelo_lia.add(layers.Conv2D(64, (3, 3), activation='relu'))

# Bloco de classificação
modelo_lia.add(layers.Flatten())
modelo_lia.add(layers.Dense(64, activation='relu'))
modelo_lia.add(layers.Dropout(0.3))
modelo_lia.add(layers.Dense(10, activation='softmax'))

modelo_lia.summary()


In [ ]:
# ── Compilação do modelo ─────────────────────────────────────────────────────
modelo_lia.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


## 🚀 Treinamento

- `validation_split=0.2` — 20 % do treino reservado para validação; o conjunto de teste permanece
  **intocado** até a avaliação final.
- **Callbacks** utilizados:
  - `EarlyStopping` — para o treino se a `val_loss` não melhorar por 3 épocas consecutivas;
  - `ReduceLROnPlateau` — reduz a taxa de aprendizado quando o treinamento estagna.
- `verbose=1` — exibe uma barra de progresso por época (menos poluído que `verbose=2`).


In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True, verbose=1
)
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1
)

# ── Treinamento ───────────────────────────────────────────────────────────────
historico = modelo_lia.fit(
    imagens_treino,
    labels_treino,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


## 📈 Curvas de Treinamento


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# ── Acurácia ─────────────────────────────────────────────────────────────────
ax1.plot(historico.history['accuracy'],     label='Treino',    linewidth=2)
ax1.plot(historico.history['val_accuracy'], label='Validação', linewidth=2)
ax1.set_title('📊 Acurácia por Época', fontsize=13)
ax1.set_xlabel('Época')
ax1.set_ylabel('Acurácia')
ax1.legend()

# ── Loss ──────────────────────────────────────────────────────────────────────
ax2.plot(historico.history['loss'],     label='Treino',    linewidth=2)
ax2.plot(historico.history['val_loss'], label='Validação', linewidth=2)
ax2.set_title('📉 Loss por Época', fontsize=13)
ax2.set_xlabel('Época')
ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.show()


## 🎯 Avaliação do Modelo

Avaliação no conjunto de **teste** (nunca visto durante o treinamento).


In [ ]:
# ── Avaliação no conjunto de teste ──────────────────────────────────────────
erro_teste, acc_teste = modelo_lia.evaluate(imagens_teste, labels_teste, verbose=0)

print(f'📉 Loss  no teste : {erro_teste:.4f}')
print(f'✅ Acurácia no teste: {acc_teste * 100:.2f}%')


In [ ]:
# ── Previsões no conjunto de teste ──────────────────────────────────────────
y_pred_prob   = modelo_lia.predict(imagens_teste, verbose=0)
y_pred_classes = np.argmax(y_pred_prob, axis=1)
y_true         = labels_teste

# ── Matriz de Confusão ────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(11, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=nomes_classes,
    yticklabels=nomes_classes,
    linewidths=0.5
)
plt.xlabel('Previsto', fontsize=12)
plt.ylabel('Real', fontsize=12)
plt.title('🔢 Matriz de Confusão — Fashion MNIST', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# ── Relatório de Classificação ───────────────────────────────────────────────
print('📋 Relatório de Classificação:\n')
print(classification_report(y_true, y_pred_classes, target_names=nomes_classes))


## �� Visualizando Previsões do Modelo

Comparação entre a **classe real** e a **classe prevista** para amostras do conjunto de teste.
Previsões **corretas** aparecem em verde; **incorretas**, em vermelho.


In [ ]:
def mostra_previsoes(images, y_real, y_prev, class_names, quantidade=12):
    """Exibe previsões do modelo com destaque para acertos e erros.

    Parameters
    ----------
    images : np.ndarray   — imagens no formato (N, H, W) ou (N, H, W, 1)
    y_real : np.ndarray   — rótulos reais
    y_prev : np.ndarray   — rótulos previstos pelo modelo
    class_names : list    — nomes das classes
    quantidade : int      — número de imagens a exibir (máx. múltiplo de 4)
    """
    quantidade = min(quantidade, len(images))
    n_cols = 4
    n_rows = (quantidade + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    axes = axes.flatten()

    for i in range(quantidade):
        axes[i].imshow(images[i].squeeze(), cmap='gray')
        axes[i].axis('off')

        classe_real = class_names[int(y_real[i])]
        classe_prev = class_names[int(y_prev[i])]
        cor = 'green' if classe_real == classe_prev else 'red'

        axes[i].set_title(
            f'Real: {classe_real}\nPrev: {classe_prev}',
            color=cor, fontsize=8
        )

    for j in range(quantidade, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('🔬 Previsões do modelo', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


mostra_previsoes(imagens_teste, y_true, y_pred_classes, nomes_classes, quantidade=12)


## 🖼️ Inferência — Imagem do Dataset

Simulação de **deploy**: selecionamos uma única imagem do conjunto de teste, exibimos o
gráfico de probabilidades das 10 classes e identificamos a classe prevista.


In [ ]:
# ── Seleciona uma imagem do conjunto de teste ────────────────────────────────
indice = 30

nova_imagem = imagens_teste[indice]
label_real  = labels_teste[indice]

# ── Previsão ─────────────────────────────────────────────────────────────────
nova_imagem_array = np.expand_dims(nova_imagem, axis=0)  # (1, 28, 28, 1)
previsoes = modelo_lia.predict(nova_imagem_array, verbose=0)
classe_prevista = int(np.argmax(previsoes, axis=1)[0])

print(f'Classe real   : {nomes_classes[int(label_real)]}')
print(f'Classe prevista: {nomes_classes[classe_prevista]}')

# ── Visualização ─────────────────────────────────────────────────────────────
fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(10, 3))

ax_img.imshow(nova_imagem.squeeze(), cmap='gray')
ax_img.axis('off')
ax_img.set_title(
    f'Real: {nomes_classes[int(label_real)]}\nPrevisto: {nomes_classes[classe_prevista]}',
    fontsize=11
)

cores = ['#1f77b4'] * 10
cores[classe_prevista] = '#d62728'  # destaca a classe prevista
ax_bar.barh(nomes_classes, previsoes[0], color=cores)
ax_bar.set_xlabel('Probabilidade')
ax_bar.set_title('🔝 Probabilidades por classe', fontsize=11)
ax_bar.invert_yaxis()

plt.tight_layout()
plt.show()


## 🖼️ Teste com Imagem Local Externa

Aqui você pode carregar uma imagem do seu computador (por exemplo, uma roupa baixada da
internet) e verificar como o modelo a classifica.

> ⚠️ **Observações importantes:**
> - O modelo espera imagens em **tons de cinza**.
> - A imagem será redimensionada para **28 × 28 pixels**.
> - Em muitas imagens da internet, o item aparece **escuro em fundo claro**; nesse caso,
>   pode ser necessário **inverter os pixels** para se aproximar do padrão visual do Fashion MNIST.


In [ ]:
# ── Detecta o ambiente de execução ──────────────────────────────────────────
from PIL import Image

try:
    from google.colab import files
    ambiente_colab = True
except ImportError:
    ambiente_colab = False

print('Executando no Google Colab?', ambiente_colab)


In [ ]:
# ── Upload da imagem no Google Colab ────────────────────────────────────────
# Execute esta célula no Colab e escolha uma imagem do seu computador.
# Fora do Colab, defina 'caminho_imagem' manualmente na próxima célula.

if ambiente_colab:
    uploaded = files.upload()
    caminho_imagem = list(uploaded.keys())[0]
    print('Arquivo enviado:', caminho_imagem)
else:
    print('Fora do Colab — defina caminho_imagem na célula seguinte.')


In [ ]:
# ── Caminho manual da imagem (fora do Colab) ────────────────────────────────
# Descomente e ajuste o caminho se necessário.

# caminho_imagem = '/content/minha_roupa.jpg'
print('Defina caminho_imagem caso não tenha feito upload na célula anterior.')


In [ ]:
# ── Pré-processamento da imagem local ───────────────────────────────────────
# Ordem: abrir → cinza → redimensionar → normalizar → inverter (opcional)

inverter_pixels = True  # False se o item já estiver claro em fundo escuro

img = Image.open(caminho_imagem).convert('L').resize((28, 28))
img_array = np.array(img).astype('float32') / 255.0

if inverter_pixels:
    img_array = 1.0 - img_array

plt.figure(figsize=(3, 3))
plt.imshow(img_array, cmap='gray')
plt.axis('off')
plt.title('Imagem processada (28 × 28)', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# ── Formata e classifica ─────────────────────────────────────────────────────
img_array_4d = np.expand_dims(np.expand_dims(img_array, axis=-1), axis=0)  # (1, 28, 28, 1)
print('Formato final da imagem:', img_array_4d.shape)

pred_local = modelo_lia.predict(img_array_4d, verbose=0)
classe_prevista_local = int(np.argmax(pred_local, axis=1)[0])

print(f'\n🏷️  Classe prevista: {nomes_classes[classe_prevista_local]}')
print('\nProbabilidades:')
for nome, prob in zip(nomes_classes, pred_local[0]):
    bar = '█' * int(prob * 30)
    print(f'  {nome:<14} {bar} {prob * 100:.1f}%')


## 🏁 Conclusão e Próximos Passos

### Resultados obtidos

| Métrica | Valor |
|---------|-------|
| Acurácia no teste | *ver saída da célula de avaliação* |
| Loss no teste | *ver saída da célula de avaliação* |

### 💡 Classes mais difíceis (comuns no Fashion MNIST)
- **Camisa** × **Camiseta/Top** × **Pulôver** — alta similaridade visual.
- **Tênis** × **Sandália** × **Bota** — classes de calçados frequentemente confundidas.

### 🚀 Próximos passos sugeridos
1. **Data augmentation** — rotação, flip horizontal, zoom — para melhorar generalização.
2. **Batch Normalization** entre as camadas convolucionais.
3. **Transfer Learning** com modelos pré-treinados (EfficientNet, MobileNet) adaptados para cinza.
4. **Quantização** do modelo para deploy em dispositivos móveis (TFLite).
5. **Explainabilidade** com Grad-CAM para visualizar o que o modelo "olha" na imagem.

---

> Se você chegou até aqui, **parabéns!** 🎆🔥  
> Você construiu uma CNN ponta a ponta — do carregamento dos dados até a inferência com uma imagem
> local — usando **TensorFlow/Keras** e o dataset **Fashion MNIST**.
